# TMJDLC — Javtiful Colab Download Smoke Test

Run top to bottom. On the first runtime, the notebook installs missing dependencies and clones TMJDLC. On later runs in the **same Colab runtime**, it skips already-installed dependencies and only updates the repository from GitHub. Your output files are kept in `/content/tmjdlc-test/output` unless you delete them.

**Note:** Colab deletes installed packages and `/content` when its runtime is factory-reset or disconnected. After a fresh runtime, setup is necessarily repeated; within a live runtime, rerunning setup only checks and skips what is already installed.

Temporary stream URLs and session headers stay in runtime memory; they are not printed or persisted.

In [ ]:
# Install only missing tools/packages. Safe to rerun in the same Colab runtime.
import importlib.util, shutil, subprocess, sys
from pathlib import Path

missing = []
for module, package in [('playwright', 'playwright>=1.50'), ('yt_dlp', 'yt-dlp>=2025.1'), ('requests', 'requests>=2.32')]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)
if missing:
    print('Installing missing Python packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('Python dependencies already installed; skipping pip.')

if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    print('Installing missing FFmpeg tools...')
    subprocess.run(['bash', '-lc', 'apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
else:
    print('FFmpeg/FFprobe already available; skipping apt install.')

# Do not enter Playwright Sync API from a notebook cell: Colab already runs asyncio.
# Detect the browser installation by inspecting Playwright's cache directory instead.
browser_cache = Path.home() / '.cache' / 'ms-playwright'
chromium_installed = browser_cache.exists() and any(browser_cache.glob('chromium-*'))
if not chromium_installed:
    print('Playwright Chromium not found; installing browser...')
    subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=True)
else:
    print('Playwright Chromium already installed; skipping browser install.')


In [ ]:
import pathlib, shutil, subprocess, json, os, sys, time
WORKDIR = pathlib.Path('/content/tmjdlc-test')
REPO_DIR = WORKDIR / 'TMJDLC'
WORKDIR.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/Telebotfaroff/TMJDLC.git'
BRANCH = 'main'

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print('First run: cloning TMJDLC...')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Existing checkout found: updating only repository files...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'FETCH_HEAD'], check=True)

%cd /content/tmjdlc-test/TMJDLC

# Install the project editable only if its distribution is not installed yet.
try:
    from importlib.metadata import distribution
    distribution('tmjdlc')
    print('TMJDLC package already installed; repository source has been updated.')
except Exception:
    print('Installing TMJDLC package for the first time...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)


## Configure the test

Default URL is the post already confirmed by the resolver workflow. Change it when testing another post.

In [ ]:
POST_URL = input('Javtiful post URL [https://javtiful.com/video/113371/dvmm-444]: ').strip() or 'https://javtiful.com/video/113371/dvmm-444'
QUALITY = input('Preferred quality (360/480/720/1080) [720]: ').strip() or '720'
if QUALITY not in {'360','480','720','1080','1440','2160'}: raise ValueError('Unsupported quality')
TIMEOUT, CAPTURE_SECONDS = 45, 20
OUTPUT_DIR = WORKDIR / 'output'
SCREENSHOT_DIR = OUTPUT_DIR / 'screenshots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
print('Post:', POST_URL, '| quality:', QUALITY + 'p')

## Resolve the stream

The resolver returns a temporary URL and session headers. This cell deliberately hides both.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tmjdlc.providers.javtiful import JavtifulStreamDetector

def field(obj, name, default=None):
    return obj.get(name, default) if isinstance(obj, dict) else getattr(obj, name, default)

# Colab/IPython runs an asyncio event loop. Playwright's Sync API must run
# in a separate worker thread to avoid the 'asyncio loop is running' error.
def resolve_stream():
    detector = JavtifulStreamDetector(
        timeout_ms=TIMEOUT * 1000,
        capture_ms=CAPTURE_SECONDS * 1000,
    )
    return detector.detect(POST_URL, preferred_quality=int(QUALITY))

with ThreadPoolExecutor(max_workers=1) as pool:
    stream_result = pool.submit(resolve_stream).result()

stream_url = field(stream_result, 'stream_url')
stream_format = field(stream_result, 'format', 'unknown')
selected_quality = field(stream_result, 'quality', QUALITY)
duration_seconds = field(stream_result, 'duration_seconds')
headers = field(stream_result, 'headers', {}) or {}
if not stream_url:
    raise RuntimeError('Resolver returned no stream URL')

print('Resolved:', stream_format, '| quality:', str(selected_quality) + 'p', '| duration:', round(duration_seconds, 2) if duration_seconds else 'unknown')
print('Session headers captured:', bool(headers), '| signed URL hidden')


## Preflight: inspect the selected HLS playlist

This checks the manifest before downloading. It reports playlist structure and duration without printing signed URLs or segment addresses. If it only exposes a short preview playlist, stop here and share the diagnostic output.

In [ ]:
import requests, re

manifest_info = {'http_status': None, 'content_type': None, 'is_master': False, 'segment_count': 0, 'playlist_duration_seconds': 0.0}
if stream_format == 'hls' or stream_url.lower().split('?', 1)[0].endswith('.m3u8'):
    try:
        response = requests.get(stream_url, headers=headers, timeout=30)
        manifest_info['http_status'] = response.status_code
        manifest_info['content_type'] = response.headers.get('content-type', '').split(';', 1)[0]
        response.raise_for_status()
        body = response.text
        manifest_info['is_master'] = '#EXT-X-STREAM-INF' in body
        manifest_info['segment_count'] = len(re.findall(r'^#EXTINF:', body, flags=re.M))
        manifest_info['playlist_duration_seconds'] = round(sum(float(x) for x in re.findall(r'^#EXTINF:([0-9.]+)', body, flags=re.M)), 2)
        manifest_info['has_endlist'] = '#EXT-X-ENDLIST' in body
        manifest_info['variant_count'] = len(re.findall(r'^#EXT-X-STREAM-INF:', body, flags=re.M))
    except Exception as exc:
        manifest_info['request_error'] = type(exc).__name__
else:
    manifest_info['note'] = 'Resolved source is not an HLS .m3u8 URL'

print('Manifest diagnostics (no signed URLs shown):')
print(json.dumps(manifest_info, indent=2))
if manifest_info.get('http_status') and manifest_info['http_status'] >= 400:
    raise RuntimeError('Manifest request failed. Do not continue to download; share the diagnostics.')
if manifest_info.get('segment_count', 0) and duration_seconds:
    ratio = manifest_info['playlist_duration_seconds'] / float(duration_seconds)
    print('Playlist/page-duration ratio:', round(ratio, 4))
    if ratio < 0.5:
        raise RuntimeError('The selected media playlist is much shorter than the page duration. This may be a preview/incorrect playlist; stop and share this diagnostic output.')


## Download with yt-dlp

Downloads and merges the HLS stream into a local media file. The signed URL is not displayed.

In [ ]:
import yt_dlp
video_path = OUTPUT_DIR / 'javtiful_video.mp4'
opts = {
 'outtmpl': str(OUTPUT_DIR / 'javtiful_video.%(ext)s'), 'format': 'best',
 'merge_output_format': 'mp4', 'noplaylist': True, 'retries': 10,
 'fragment_retries': 10, 'skip_unavailable_fragments': False,
 'concurrent_fragment_downloads': 4, 'http_headers': dict(headers),
 'quiet': True, 'no_warnings': False,
 'progress_hooks': [lambda d: print('Download:', d.get('status'), d.get('_percent_str','').strip(), d.get('_speed_str','').strip(), 'ETA', d.get('_eta_str','').strip()) if d.get('status') == 'downloading' else None]
}
with yt_dlp.YoutubeDL(opts) as ydl: ydl.download([stream_url])
candidates = [p for p in OUTPUT_DIR.glob('javtiful_video.*') if p.is_file() and p.suffix.lower() not in {'.part','.ytdl','.json'}]
if not candidates: raise FileNotFoundError('No downloaded media file found')
video_path = max(candidates, key=lambda p:p.stat().st_size)
print('Downloaded:', video_path.name, '| MiB:', round(video_path.stat().st_size/1048576,2))

## Verify with FFprobe

In [ ]:
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size,format_name:stream=codec_type,codec_name,width,height','-of','json',str(video_path)],capture_output=True,text=True)
if probe.returncode: raise RuntimeError('FFprobe failed: '+probe.stderr[-1200:])
probe_data = json.loads(probe.stdout)
print(json.dumps(probe_data, indent=2))
# Reject incomplete/preview downloads before screenshots or uploads.
expected_duration = float(duration_seconds) if duration_seconds else None
actual_duration = float(probe_data.get('format', {}).get('duration') or 0)
print('Expected page duration:', round(expected_duration, 2) if expected_duration else 'unknown')
print('Downloaded duration:', round(actual_duration, 2))
if expected_duration and actual_duration < expected_duration * 0.85:
    raise RuntimeError('Downloaded file is shorter than expected; likely a preview or incomplete stream. Do not generate screenshots or upload it.')


## Generate four screenshots with FFmpeg

In [ ]:
try: duration = float(probe_data.get('format',{}).get('duration'))
except (TypeError,ValueError): duration = None
timestamps = [max(1,int(duration*f)) for f in (.15,.35,.60,.85)] if duration and duration > 8 else [10,30,60,90]
generated = []
for i, seconds in enumerate(timestamps,1):
    out = SCREENSHOT_DIR / f'frame_{i}.jpg'
    result = subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(seconds),'-i',str(video_path),'-frames:v','1','-q:v','2',str(out)],capture_output=True,text=True)
    if result.returncode == 0 and out.exists() and out.stat().st_size: generated.append(out)
    else: print('Frame failed at', seconds, 'seconds:', result.stderr[-400:])
print('Screenshots generated:', len(generated))
if not generated: raise RuntimeError('No screenshots generated')

## Preview screenshots

In [ ]:
from IPython.display import display, Image
for p in generated:
    print(p.name)
    display(Image(filename=str(p)))

## Download test outputs

Creates one ZIP containing the MP4 and generated screenshots.

In [ ]:
import zipfile
from google.colab import files
zip_path = WORKDIR / 'tmjdlc-smoke-test-output.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(video_path, arcname=video_path.name)
    for p in generated: z.write(p, arcname='screenshots/'+p.name)
print('Archive size MiB:', round(zip_path.stat().st_size/1048576,2))
files.download(str(zip_path))

## Optional cleanup

Run only after downloading the ZIP.

In [ ]:
# Uncomment to remove temporary files:
# shutil.rmtree(WORKDIR, ignore_errors=True)